# ComfyUI on Google Colab (+ ComfyUI-Manager)

## 使い方
1. **ランタイム → ランタイムのタイプを変更 → GPU（T4 など）** を選択
2. セル **[1]** → **[2]** → **[3]** を順に実行
3. セル [3] の出力に出る `https://xxxx.trycloudflare.com` が ComfyUI の URL

セル [3] は ComfyUI を動かし続けるため **実行したままに**してください。停止すると URL も切れます。

---

### 修正履歴
旧版は起動時に `ModuleNotFoundError: No module named 'comfy_aimdo'` で失敗していました。

| 問題 | 原因 | 対処 |
|---|---|---|
| `comfy_aimdo` が無い | セル[1]が `requirements.txt` を使わず数年前の手書きリストを入れていた。現行必須の16個（`comfy-aimdo`, `comfyui-frontend-package`, `comfy-kitchen`, `av`, `pydantic`, `SQLAlchemy` …）が欠落 | `pip install -r requirements.txt` に変更 |
| torch が壊れる | `--index-url .../cu121` で torch を強制再インストールし、Colab の CUDA 版を破壊 | torch 系は除外し Colab の版をそのまま使う |
| SD1.5 が落ちない | `runwayml/stable-diffusion-v1-5` が HuggingFace から削除済み（HTTP 401） | `Comfy-Org/stable-diffusion-v1-5-archive` に変更 |
| `>=` がシェルリダイレクト | `!pip3 install transformers>=4.28.1` の `>` がファイル作成になり、バージョン指定が無視されていた | requirements.txt 経由にして解消 |

`blake3` / `alembic` を個別に入れる回避セルは不要になったため削除しました。

In [ ]:
#@title [1] 環境セットアップ（ComfyUI 本体 + ComfyUI-Manager）

USE_GOOGLE_DRIVE    = False  #@param {type:"boolean"}
UPDATE_COMFY_UI     = True   #@param {type:"boolean"}
USE_COMFYUI_MANAGER = True   #@param {type:"boolean"}

import os

if USE_GOOGLE_DRIVE:
    # Drive に置くと再起動後もモデルが残るが、読み書きが遅く Drive 容量も消費する
    from google.colab import drive
    drive.mount('/content/drive')
    WORKSPACE = "/content/drive/MyDrive/ComfyUI"
else:
    WORKSPACE = "/content/ComfyUI"
print("WORKSPACE:", WORKSPACE)

if not os.path.exists(WORKSPACE):
    print("-= Initial setup ComfyUI =-")
    !git clone https://github.com/comfyanonymous/ComfyUI $WORKSPACE
elif UPDATE_COMFY_UI:
    print("-= Updating ComfyUI =-")
    !git -C $WORKSPACE pull

os.chdir(WORKSPACE)

# ---------------------------------------------------------------------------
# 依存関係のインストール
#
# requirements.txt をそのまま使うこと。旧版は手書きのパッケージ一覧を入れていたため
# 現行 ComfyUI の必須依存（comfy-aimdo, comfyui-frontend-package, comfy-kitchen,
# av, pydantic, SQLAlchemy など16個）が抜け、起動時に
#   ModuleNotFoundError: No module named 'comfy_aimdo'
# で失敗していた。
#
# torch / torchvision / torchaudio だけ除外するのは、Colab にプリインストール済みの
# CUDA 版 torch が CPU 版や別の CUDA ビルドに差し替わるのを防ぐため。
# （"torchsde" は直後が "s" なのでこの正規表現には一致せず、ちゃんと入る）
# ---------------------------------------------------------------------------
print("-= Install dependencies =-")
!grep -vE '^(torch|torchvision|torchaudio)([=<>!~ ]|$)' requirements.txt > /tmp/comfy_req.txt
!pip install -q -r /tmp/comfy_req.txt

if USE_COMFYUI_MANAGER:
    print("-= Setup ComfyUI-Manager =-")
    !mkdir -p custom_nodes
    !if [ ! -d custom_nodes/comfyui-manager ]; then \
        git clone https://github.com/Comfy-Org/ComfyUI-Manager custom_nodes/comfyui-manager; \
      else \
        git -C custom_nodes/comfyui-manager pull; \
      fi
    !pip install -q -r custom_nodes/comfyui-manager/requirements.txt

import torch
print()
print("torch:", torch.__version__, "| CUDA:", torch.cuda.is_available())
if not torch.cuda.is_available():
    print("!!! GPUが有効ではありません → ランタイム → ランタイムのタイプを変更 → GPU")

### モデルのダウンロード

既定では SD1.5 と VAE を取得します。他のモデルは必要なものだけコメントを外してください。

In [ ]:
#@title [2] モデルのダウンロード（必要なものだけコメントを外す）
import os
os.chdir(WORKSPACE)

# --- SD1.5 ---
# 注意: 旧版が使っていた runwayml/stable-diffusion-v1-5 は HuggingFace から
#       削除済み（HTTP 401）で wget が失敗する。Comfy-Org のアーカイブを使うこと。
!wget -c -nv https://huggingface.co/Comfy-Org/stable-diffusion-v1-5-archive/resolve/main/v1-5-pruned-emaonly-fp16.safetensors -P ./models/checkpoints/

# --- VAE ---
!wget -c -nv https://huggingface.co/stabilityai/sd-vae-ft-mse-original/resolve/main/vae-ft-mse-840000-ema-pruned.safetensors -P ./models/vae/

### SDXL
### ワークフロー例: https://comfyanonymous.github.io/ComfyUI_examples/sdxl/
#!wget -c https://huggingface.co/stabilityai/stable-diffusion-xl-base-1.0/resolve/main/sd_xl_base_1.0.safetensors -P ./models/checkpoints/
#!wget -c https://huggingface.co/stabilityai/stable-diffusion-xl-refiner-1.0/resolve/main/sd_xl_refiner_1.0.safetensors -P ./models/checkpoints/

### FLUX.1 schnell
#!wget -c https://huggingface.co/Comfy-Org/flux1-schnell/resolve/main/flux1-schnell-fp8.safetensors -P ./models/checkpoints/

### SD2.1
#!wget -c https://huggingface.co/stabilityai/stable-diffusion-2-1-base/resolve/main/v2-1_512-ema-pruned.safetensors -P ./models/checkpoints/

### ControlNet (SD1.5)
#!wget -c https://huggingface.co/comfyanonymous/ControlNet-v1-1_fp16_safetensors/resolve/main/control_v11p_sd15_canny_fp16.safetensors -P ./models/controlnet/
#!wget -c https://huggingface.co/comfyanonymous/ControlNet-v1-1_fp16_safetensors/resolve/main/control_v11p_sd15_openpose_fp16.safetensors -P ./models/controlnet/

### アップスケーラ
#!wget -c https://github.com/xinntao/Real-ESRGAN/releases/download/v0.1.0/RealESRGAN_x4plus.pth -P ./models/upscale_models/

### LoRA
#!wget -c https://huggingface.co/stabilityai/stable-diffusion-xl-base-1.0/resolve/main/sd_xl_offset_example-lora_1.0.safetensors -P ./models/loras/

print("--- models/checkpoints ---")
!ls -lh ./models/checkpoints/
print("--- models/vae ---")
!ls -lh ./models/vae/

### ComfyUI を起動（cloudflared / 推奨）

下のセルを実行し、出力される `https://xxxx.trycloudflare.com` を開いてください。
ComfyUI-Manager は画面のサイドバーの **Manager** ボタンから使えます。

> このURLは認証なしの公開URLです。リンクを知っている人は誰でもアクセスできます。

In [ ]:
#@title [3] ComfyUI を起動（cloudflared / 推奨）
# このセルは実行したままにしておくこと。停止すると ComfyUI も URL も落ちます。
import os
os.chdir(WORKSPACE)

!wget -q -P ~ https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64.deb
!dpkg -i ~/cloudflared-linux-amd64.deb > /dev/null 2>&1 || apt-get -y -f install > /dev/null 2>&1

import subprocess, threading, time, socket, re

PORT = 8188

def tunnel(port):
    while True:                       # ComfyUI が listen するまで待つ
        time.sleep(0.5)
        with socket.socket() as s:
            if s.connect_ex(("127.0.0.1", port)) == 0:
                break
    print("\nComfyUI の起動を確認。cloudflared トンネルを開きます...\n")
    # cloudflared は URL を stderr に出すので stdout にまとめて正規表現で拾う
    # （旧版は "trycloudflare.com " という末尾スペース付き固定文字列で探しており脆かった）
    p = subprocess.Popen(
        ["cloudflared", "tunnel", "--url", f"http://127.0.0.1:{port}"],
        stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1,
    )
    for line in p.stdout:
        m = re.search(r"https://[-\w]+\.trycloudflare\.com", line)
        if m:
            print("=" * 68)
            print("  ComfyUI の URL:  " + m.group(0))
            print("=" * 68)

threading.Thread(target=tunnel, args=(PORT,), daemon=True).start()

!python main.py --port 8188 --dont-print-server

### 予備1: localtunnel で起動

cloudflared がうまくいかない場合の代替です。表示されるパスワード欄には、一緒に出力される IP を入力してください。

In [ ]:
#@title [予備1] localtunnel で起動
import os
os.chdir(WORKSPACE)

!npm install -g localtunnel

import subprocess, threading, time, socket, urllib.request

PORT = 8188

def tunnel(port):
    while True:
        time.sleep(0.5)
        with socket.socket() as s:
            if s.connect_ex(("127.0.0.1", port)) == 0:
                break
    print("\nComfyUI の起動を確認。localtunnel を開きます...\n")
    ip = urllib.request.urlopen("https://ipv4.icanhazip.com").read().decode().strip()
    print("localtunnel のパスワード（= このIP）:", ip)
    p = subprocess.Popen(["lt", "--port", str(port)], stdout=subprocess.PIPE, text=True, bufsize=1)
    for line in p.stdout:
        print(line, end="")

threading.Thread(target=tunnel, args=(PORT,), daemon=True).start()

!python main.py --port 8188 --dont-print-server

### 予備2: Colab の iframe で開く

上の2つがどちらも駄目な場合のみ。Colab の iframe は WebSocket を遮断するため、生成中のライブプレビューなど一部機能が動きません。

In [ ]:
#@title [予備2] Colab iframe で起動
import os
os.chdir(WORKSPACE)

import threading, time, socket

PORT = 8188

def iframe_thread(port):
    while True:
        time.sleep(0.5)
        with socket.socket() as s:
            if s.connect_ex(("127.0.0.1", port)) == 0:
                break
    from google.colab import output
    output.serve_kernel_port_as_iframe(port, height=1024)
    print("別ウィンドウで開く場合はこちら:")
    output.serve_kernel_port_as_window(port)

threading.Thread(target=iframe_thread, args=(PORT,), daemon=True).start()

!python main.py --port 8188 --dont-print-server